In [0]:
# Databricks notebook source
# Task 04: Exportar a CosmosDB usando Delta Change Data Feed (CDF)

# COMMAND ----------
# Configuracion base (constantes)
UC_CATALOG = "parts"
UC_SCHEMA = "bronze"

# Config file (DBFS)
CONFIG_PATH = "dbfs:/config/cosmos_export.json"

# COMMAND ----------
import json
from datetime import datetime
from pyspark.sql import functions as F

def log(msg: str):
    ts = datetime.utcnow().isoformat()
    print(f"{ts} | {msg}")

def load_config(path: str):
    try:
        raw = dbutils.fs.head(path, 1_000_000)
    except Exception as e:
        raise ValueError(f"No se pudo leer config en {path}") from e
    try:
        return json.loads(raw)
    except Exception as e:
        raise ValueError(f"Config JSON invalido en {path}") from e

# COMMAND ----------
cfg = load_config(CONFIG_PATH)

catalog = (cfg.get("catalog") or UC_CATALOG).strip()
schema = (cfg.get("schema") or UC_SCHEMA).strip()
table_pattern = (cfg.get("table_pattern") or "*").strip()
if table_pattern == "%":
    table_pattern = "*"

# Cosmos config
cosmos_endpoint = (cfg.get("cosmos_endpoint") or "").strip()
cosmos_key_scope = (cfg.get("cosmos_key_scope") or "").strip()
cosmos_key_key = (cfg.get("cosmos_key_key") or "").strip()
cosmos_database = (cfg.get("cosmos_database") or "").strip()
cosmos_container = (cfg.get("cosmos_container") or "").strip()
partition_key = (cfg.get("partition_key") or "/part_number").strip()

if not cosmos_endpoint or not cosmos_key_scope or not cosmos_key_key or not cosmos_database or not cosmos_container:
    raise ValueError("Faltan parametros de Cosmos en config (endpoint/scope/key/db/container).")

cosmos_key = dbutils.secrets.get(scope=cosmos_key_scope, key=cosmos_key_key)

# CDF checkpoint (por tabla)
checkpoint_base = (cfg.get("checkpoint_base") or f"abfss://blobstorage@datablobstorage001.dfs.core.windows.net/bronze/_checkpoints/cosmos_cdf").strip()

# COMMAND ----------
# Forzar catalog/schema
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

tables = (
    spark.sql(f"SHOW TABLES IN {catalog}.{schema} LIKE '{table_pattern}'")
    .select("tableName")
    .collect()
)
table_names = [f"{catalog}.{schema}.{r['tableName']}" for r in tables]

if not table_names:
    raise ValueError(f"No se encontraron tablas con pattern '{table_pattern}' en {catalog}.{schema}.")

log(f"Tablas a procesar: {table_names}")

# COMMAND ----------
for table_name in table_names:
    if table_name.endswith("autoloader_logs"):
        continue

    log(f"Procesando CDF de {table_name}")

    # Checkpoint por tabla
    table_ckpt = f"{checkpoint_base}/{table_name.replace('.', '_')}"

    # Si no hay checkpoint, hacer export inicial batch y luego iniciar CDF desde la version actual
    def checkpoint_exists(path: str) -> bool:
        try:
            dbutils.fs.ls(path)
            return True
        except Exception:
            return False

    if not checkpoint_exists(table_ckpt):
        log(f"No existe checkpoint. Haciendo export inicial batch de {table_name}")
        base_df = spark.table(table_name)
        if "part_number" not in base_df.columns:
            raise ValueError(f"{table_name} no tiene part_number")
        base_df = base_df.withColumn("id", F.col("part_number").cast("string"))
        base_df = base_df.withColumn("sheet", F.lit(table_name.split(".")[-1]))

        (base_df.write
            .format("cosmos.oltp")
            .option("spark.cosmos.accountEndpoint", cosmos_endpoint)
            .option("spark.cosmos.accountKey", cosmos_key)
            .option("spark.cosmos.database", cosmos_database)
            .option("spark.cosmos.container", cosmos_container)
            .option("spark.cosmos.write.strategy", "ItemOverwrite")
            .option("spark.cosmos.write.bulk.enabled", "true")
            .mode("append")
            .save())

        # Obtener version actual y primera version con CDF habilitado
        history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
        latest_version = history_df.select("version").limit(1).collect()[0]["version"]

        cdf_enabled = (
            history_df
            .filter(
                (F.col("operation") == "SET TBLPROPERTIES") &
                (
                    F.col("operationParameters").getItem("properties").contains("delta.enableChangeDataFeed")
                    | (F.col("operationParameters").getItem("delta.enableChangeDataFeed") == "true")
                )
            )
            .select("version")
            .orderBy(F.col("version").asc())
            .limit(1)
            .collect()
        )
        if cdf_enabled:
            cdf_start = cdf_enabled[0]["version"]
        else:
            # Si no hay SET TBLPROPERTIES, puede estar habilitado desde CREATE TABLE
            cdf_from_create = (
                history_df
                .filter(F.col("operation").isin("CREATE TABLE", "CREATE TABLE AS SELECT"))
                .select("version", "operationParameters")
                .collect()
            )
            cdf_start = None
            for row in cdf_from_create:
                params = row["operationParameters"]
                if params and (
                    "delta.enableChangeDataFeed" in params
                    and str(params["delta.enableChangeDataFeed"]).lower() == "true"
                ):
                    cdf_start = row["version"]
                    break

        if cdf_start is None:
            # Habilita CDF para futuros cambios y omite streaming en esta corrida
            log(f"WARNING: No se encontro cdf_start. Habilitando CDF y omitiendo streaming por ahora para {table_name}.")
            spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
            continue
        else:
            # Inicia CDF desde la primera version con CDF o desde la siguiente version posterior al snapshot exportado
            start_version = max(cdf_start, latest_version + 1)
            if start_version > latest_version:
                log(f"INFO: start_version {start_version} > latest_version {latest_version}. Omitiendo streaming por ahora para {table_name}.")
                continue
            log(f"Inicial export OK. CDF start_version={start_version} (cdf_start={cdf_start}, latest={latest_version})")
    else:
        start_version = None
        log(f"Checkpoint existente. Continuando CDF desde checkpoint: {table_ckpt}")

    # Leer cambios desde la última ejecución
    reader = (
        spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
    )
    if start_version is not None:
        reader = reader.option("startingVersion", str(start_version))
    cdf_df = reader.table(table_name)

    # Solo inserts y updates; omite deletes
    cdf_df = cdf_df.filter(F.col("_change_type").isin("insert", "update_postimage"))

    # Mapea campos al documento de Cosmos
    def build_doc(df):
        if "part_number" not in df.columns:
            raise ValueError(f"{table_name} no tiene part_number")
        # id requerido en Cosmos
        df = df.withColumn("id", F.col("part_number").cast("string"))
        df = df.withColumn("sheet", F.lit(table_name.split(".")[-1]))
        # Metadatos mínimos
        return df

    doc_df = build_doc(cdf_df)

    query = (
        doc_df.writeStream
        .format("cosmos.oltp")
        .option("spark.cosmos.accountEndpoint", cosmos_endpoint)
        .option("spark.cosmos.accountKey", cosmos_key)
        .option("spark.cosmos.database", cosmos_database)
        .option("spark.cosmos.container", cosmos_container)
        .option("spark.cosmos.write.strategy", "ItemOverwrite")
        .option("spark.cosmos.write.bulk.enabled", "true")
        .option("checkpointLocation", table_ckpt)
        .outputMode("append")
        .trigger(availableNow=True)
        .start()
    )

    query.awaitTermination()
    log(f"Export finalizado para {table_name}")

log("Task 04 completed.")
